In [2]:
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

DATABASE_DIR = Path("../../database")

diseases = pd.read_csv(DATABASE_DIR / "diseases.csv")
symptoms = pd.read_csv(DATABASE_DIR / "symptoms.csv")
disease_symptoms = pd.read_csv(DATABASE_DIR / "disease_symptoms.csv")
matrix = pd.read_csv(DATABASE_DIR / "disease_symptom_matrix.csv")

print("Diseases:", diseases.shape)
print("Symptoms:", symptoms.shape)
print("Disease-Symptom relationships:", disease_symptoms.shape)
print("Disease-Symptom matrix:", matrix.shape)

Diseases: (34, 7)
Symptoms: (151, 8)
Disease-Symptom relationships: (243, 4)
Disease-Symptom matrix: (34, 153)


In [3]:
symptoms_per_disease = (
    disease_symptoms
    .groupby("disease_id")
    .size()
    .sort_values(ascending=False)
)

print(symptoms_per_disease)

disease_id
D001    12
D005    11
D002     9
D004     9
D007     9
D011     9
D014     8
D017     8
D009     8
D033     8
D018     7
D013     7
D003     7
D032     7
D022     7
D024     7
D020     7
D006     7
D034     7
D031     7
D023     7
D026     6
D008     6
D015     6
D012     6
D016     6
D029     6
D025     6
D019     6
D021     6
D030     6
D010     5
D027     5
D028     5
dtype: int64


In [4]:
symptoms_per_disease_named = (
    disease_symptoms
    .groupby("disease_id")
    .size()
    .reset_index(name="symptom_count")
    .merge(
        diseases[["disease_id", "disease_name"]],
        on="disease_id",
        how="left"
    )
    .sort_values("symptom_count", ascending=False)
)

display(symptoms_per_disease_named)

,disease_id,symptom_count,disease_name
0,D001,12,Diabetes
4,D005,11,Depression and Anxiety Disorder
1,D002,9,Thyroid Disorders
3,D004,9,Stroke
6,D007,9,Liver Disease
10,D011,9,COVID-19/Flu
13,D014,8,Jaundice (Hepatitis)
16,D017,8,Type 2 Diabetes
8,D009,8,Tuberculosis (TB)
32,D033,8,Chikungunya


In [5]:
symptoms_per_disease_count = (
    disease_symptoms
    .groupby("symptom_id")
    .size()
    .reset_index(name="disease_count")
    .merge(
        symptoms[["symptom_id", "symptom_name"]],
        on="symptom_id",
        how="left"
    )
    .sort_values("disease_count", ascending=False)
)

display(symptoms_per_disease_count.head(20))

,symptom_id,disease_count,symptom_name
3,S004,15,Tiredness or fatigue
4,S005,7,Shortness of breath
72,S073,7,Loss of appetite
130,S133,6,Fever
73,S074,6,Headache
57,S058,6,Nausea
49,S050,5,Vomiting
9,S010,4,Blurred vision
82,S083,4,Cough
37,S038,4,Chest pain or discomfort


In [6]:
least_common_symptoms = (
    symptoms_per_disease_count
    .sort_values("disease_count", ascending=True)
)

display(least_common_symptoms.head(20))

,symptom_id,disease_count,symptom_name
13,S014,1,Fast or irregular heartbeat
12,S013,1,Weight change
11,S012,1,Cold intolerance
10,S011,1,Heat intolerance
0,S001,1,Increased thirst
1,S002,1,Increased hunger
45,S046,1,Coughing up blood or sputum
46,S047,1,Night sweats
21,S022,1,Sudden dizziness
24,S025,1,Facial drooping


In [7]:
# Count the number of positive symptoms for each disease
matrix_symptom_columns = [
    col for col in matrix.columns
    if col not in ["disease_id", "disease_name"]
]

matrix["symptom_count"] = matrix[matrix_symptom_columns].sum(axis=1)

disease_profile = (
    matrix[["disease_id", "disease_name", "symptom_count"]]
    .sort_values("symptom_count", ascending=False)
)

display(disease_profile)

/tmp/ipykernel_31050/1039773690.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matrix["symptom_count"] = matrix[matrix_symptom_columns].sum(axis=1)


,disease_id,disease_name,symptom_count
0,D001,Diabetes,12
4,D005,Depression and Anxiety Disorder,11
1,D002,Thyroid Disorders,9
3,D004,Stroke,9
6,D007,Liver Disease,9
10,D011,COVID-19/Flu,9
13,D014,Jaundice (Hepatitis),8
16,D017,Type 2 Diabetes,8
8,D009,Tuberculosis (TB),8
32,D033,Chikungunya,8


In [8]:
print("Disease count:", diseases["disease_id"].nunique())
print("Symptom count:", symptoms["symptom_id"].nunique())

print("\nDisease-symptom relationships:")
print(len(disease_symptoms))

print("\nAverage symptoms per disease:")
print(
    len(disease_symptoms) /
    diseases["disease_id"].nunique()
)

Disease count: 34
Symptom count: 151

Disease-symptom relationships:
243

Average symptoms per disease:
7.147058823529412


In [9]:
symptom_frequency = (
    disease_symptoms
    .groupby("symptom_id")
    .size()
    .sort_values(ascending=False)
)

print("Most common symptoms:")
print(symptom_frequency.head(20))

print("\nLeast common symptoms:")
print(symptom_frequency.tail(20))

Most common symptoms:
symptom_id
S004    15
S005     7
S073     7
S133     6
S074     6
S058     6
S050     5
S010     4
S083     4
S038     4
S007     4
S034     3
S003     3
S048     3
S070     3
S138     3
S136     3
S067     3
S066     3
S088     2
dtype: int64

Least common symptoms:
symptom_id
S117    1
S114    1
S115    1
S116    1
S128    1
S127    1
S122    1
S123    1
S121    1
S132    1
S130    1
S131    1
S135    1
S141    1
S143    1
S145    1
S148    1
S149    1
S150    1
S151    1
dtype: int64
